<a href="https://colab.research.google.com/github/gopijha021/solar-tracking-system/blob/main/Snippets_Importing_libraries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing a library that is not in Colaboratory

To import a library that's not in Colaboratory by default, you can use `!pip install` or `!apt-get install`.

In [ ]:
!pip install matplotlib-venn

In [ ]:
!apt-get -qq install -y libfluidsynth1

# Install 7zip reader [libarchive](https://pypi.python.org/pypi/libarchive)

In [ ]:
# https://pypi.python.org/pypi/libarchive
!apt-get -qq install -y libarchive-dev && pip install -U libarchive
import libarchive

# Install GraphViz & [PyDot](https://pypi.python.org/pypi/pydot)

In [ ]:
# https://pypi.python.org/pypi/pydot
!apt-get -qq install -y graphviz && pip install pydot
import pydot

# Install [cartopy](http://scitools.org.uk/cartopy/docs/latest/)

In [ ]:
!pip install cartopy
import cartopy

In [10]:
import numpy as np
import h5py
from astropy.io import fits
import glob
from tqdm import tqdm
from ipdb import set_trace as stop
from skimage.registration import phase_cross_correlation as register_translation
import os

# Dataset configurations
datasets = ['continuum4170', 'Gband', 'CaK', 'Mgb2']  # 'Halpha' can be added later
pixel_size = [0.18, 0.18, 0.18, 0.18, 0.09]
cadences = [2.11*14, 2.11*14, 2.11*14, 4.22*7, 0.99*30]

def generate_dataset(output_file, n_patches_per_image, n_images, border=200, n_pixel=64):
    n_patches = n_images * n_patches_per_image

    print(f"Generating file {output_file}...")
    print(f" - N. images : {n_images}")
    print(f" - N. patches per image : {n_patches_per_image}")
    print(f" - N. patches total : {n_patches}")

    index_datasets = np.random.randint(low=0, high=len(datasets), size=n_images)

    # Collect file lists for each dataset
    files = []
    for ds in datasets:
        path = f'/net/nas/proyectos/fis/aasensio/deep_learning/deepvel_jess/{ds}/*.fits'
        matched = glob.glob(path)
        matched.sort()
        files.append(matched)

    # Output file path
    file_path = f'/net/nas/proyectos/fis/aasensio/deep_learning/deepvel_jess/{output_file}'
    os.makedirs(os.path.dirname(file_path), exist_ok=True)

    # Safely create the HDF5 file
    with h5py.File(file_path, 'w') as f:
        db_im = f.create_dataset("images", (n_patches, 2, n_pixel, n_pixel), dtype=np.float32)

        loop = 0
        for i in tqdm(range(n_images)):
            ds_files = files[index_datasets[i]]
            if len(ds_files) < 2:
                continue  # Not enough FITS files for comparison

            index = np.random.randint(low=0, high=len(ds_files) - 1)

            x0 = np.random.randint(low=border, high=1000 - border, size=n_patches_per_image)
            y0 = np.random.randint(low=border, high=1000 - border, size=n_patches_per_image)

            with fits.open(ds_files[index]) as f0, fits.open(ds_files[index + 1]) as f1:
                for j in range(n_patches_per_image):
                    im0 = f0[0].data[x0[j]:x0[j]+n_pixel, y0[j]:y0[j]+n_pixel]
                    im1 = f1[0].data[x0[j]:x0[j]+n_pixel, y0[j]:y0[j]+n_pixel]

                    # Register images
                    shift, error, diffphase = register_translation(im0, im1)
                    shift = [int(s) for s in shift]
                    im1 = np.roll(im1, shift, axis=(0, 1))

                    db_im[loop, 0, :, :] = im0
                    db_im[loop, 1, :, :] = im1
                    loop += 1

    print(" Dataset generation complete.")

# Entry point
if __name__ == '__main__':
    n_patches_per_image = 10
    n_images = 500
    border = 200
    n_pixel = 64

    generate_dataset('validation.h5', n_patches_per_image, n_images, border, n_pixel)


Generating file validation.h5...
 - N. images : 500
 - N. patches per image : 10
 - N. patches total : 5000


100%|██████████| 500/500 [00:00<00:00, 1103764.21it/s]

 Dataset generation complete.
